In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything, report_split_stats
from util import mask_crop as mask_crop_fn
from validate import val_model_stable as val_model

# ============================================================
# EXPERIMENT CONFIG
# ============================================================
EXP_NAME = "cond_pretext_no_memorize" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed_everything(42)

# ============================================================
# HELPER: VALIDATION WRAPPER
# ============================================================
class ValLoaderWrapper:
    def __init__(self, loader):
        self.loader = loader
        self.dataset = loader.dataset
    def __iter__(self):
        for batch in self.loader:
            yield batch[:3]
    def __len__(self):
        return len(self.loader)

# ============================================================
# 1. DATASET CLASS
# ============================================================
class QSM_RAM_Dataset(Dataset):
    def __init__(self, nii_dir, seg_dir, mask_crop_fn, clinical_dict, label_map, limit=None, cache_path=None, load_cache=False):
        self.samples = []
        self.volumes = {} 
        self.clinical_dict = clinical_dict
        self.label_map = label_map
        self.transform = None   
        self.train_mode = False 
        
        if load_cache and cache_path and os.path.exists(cache_path):
            print(f">>> Loading preprocessed data from cache: {cache_path}...")
            cached_data = torch.load(cache_path)
            self.volumes = cached_data['volumes']
            self.samples = cached_data['samples']
            return 

        all_potential = [f for f in os.listdir(nii_dir) if f.startswith('qsm_') and f.endswith('.nii.gz')]
        loaded_count = 0
        for f in tqdm(all_potential, desc="Caching Volumes"):
            if limit and loaded_count >= limit: break
            try:
                sub_id = int(f.split('_')[1])
                case_id = f"{sub_id:02d}"
                mask_path = os.path.join(seg_dir, f'seg_{case_id}.nii.gz')
                if not os.path.exists(mask_path): continue
                
                raw_data = nib.load(os.path.join(nii_dir, f)).get_fdata()
                mask_data = nib.load(mask_path).get_fdata()
                mask_data[mask_data <= 2] = 0
                binary_mask = (mask_data > 0).astype(np.uint8)
                
                img = mask_crop_fn(raw_data, mask_data, (72, 64, 64)) / 1000.0
                m_patch = mask_crop_fn(binary_mask, mask_data, (72, 64, 64))
                
                if img.shape != (72, 64, 64): continue
                brain_indices = m_patch > 0
                if np.any(brain_indices):
                    img = (img - np.mean(img[brain_indices])) / (np.std(img[brain_indices]) + 1e-8)
                
                processed_vol = np.transpose(np.clip(img, -5.0, 5.0), (1, 2, 0)).astype(np.float32)
                self.volumes[sub_id] = processed_vol
                
                actual_label = self.label_map.get(sub_id, -1)
                for slice_idx in range(72):
                    self.samples.append({'sub_id': sub_id, 'slice_idx': slice_idx, 'label': actual_label})
                loaded_count += 1
            except Exception: continue
        
        if cache_path:
            torch.save({'volumes': self.volumes, 'samples': self.samples}, cache_path)

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, index):
        meta = self.samples[index]
        vol = self.volumes[meta['sub_id']]
        img = vol[:, :, meta['slice_idx']]
        img_tensor = torch.from_numpy(img).unsqueeze(0) 
        if self.train_mode and self.transform:
            img_tensor = self.transform(img_tensor)
        img_tensor = img_tensor / 5.0 
        clin_data = self.clinical_dict.get(str(meta['sub_id']))
        clin_vec = torch.tensor(clin_data, dtype=torch.float32) if clin_data is not None else torch.zeros(getattr(self, 'clin_dim', 11))
        return img_tensor, clin_vec, int(meta['label']), index

# ============================================================
# 2. MODELS
# ============================================================
class QSMDecoder(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        # Input is (B, 512, 1, 1)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(feat_dim, 256, 4, 1, 0), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.decoder(x)

class ResNetWrapper(nn.Module):
    def __init__(self, model, clinical_dim):
        super().__init__()
        self.base_model = model
        self.feat_dim = model.fc.in_features
        self.base_model.fc = nn.Identity()
        self.fusion = nn.Sequential(
            nn.Linear(self.feat_dim + clinical_dim, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, 2)
        )
    def forward(self, x, clinical_vec):
        feats = self.base_model(x)
        logits = self.fusion(torch.cat([feats, clinical_vec], dim=1))
        return logits, feats.view(feats.size(0), self.feat_dim, 1, 1)

# ============================================================
# 3. RUNTIME & CV
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'

cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', 'Physician', ' pre op levadopa equivalent dose (mg)', ' Location', ' Target', ' Test medication status', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}

motor_df = filter_data(file_dir, all_needed_cols, True)
motor_df[' ON (pre-dbs updrs)'] = pd.to_numeric(motor_df[' ON (pre-dbs updrs)'], errors='coerce')
motor_df[' OFF meds ON stim 6mo'] = pd.to_numeric(motor_df[' OFF meds ON stim 6mo'], errors='coerce')
motor_df = motor_df.dropna(subset=[' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo'])

improvement_ratios = (motor_df[' ON (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' ON (pre-dbs updrs)']
label_map = {int(row['CORNELL ID']): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) for _, row in motor_df.iterrows()}

full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim

all_cached_ids = set(full_dataset.volumes.keys())
labeled_subs = np.array(list(all_cached_ids & set(label_map.keys())))
unlabeled_ids = np.array(list(all_cached_ids - set(label_map.keys())))
sub_labels = np.array([label_map[sid] for sid in labeled_subs])

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))
])

all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(labeled_subs, sub_labels)):
    train_subs, val_subs = labeled_subs[t_p_idx], labeled_subs[v_p_idx]
    report_split_stats(train_subs, val_subs, motor_df)
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    
    t_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in train_subs]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in val_subs]
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in pt_subs]

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    train_labels = [label_map[full_dataset.samples[i]['sub_id']] for i in t_idx]
    class_weights = 1. / torch.tensor(np.bincount(train_labels), dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    print(f"\n>>> Split {split} | PT Subs: {len(pt_subs)} | Val Subs: {len(val_subs)}")

    # --- A. CONDITIONAL PRETRAINING ---
    base_resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)
    
    # Optimizer: Only update Backbone + Decoder. Skip 'fusion' to avoid identity memorization.
    optimizer_pt = torch.optim.Adam(list(model.base_model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()

    full_dataset.train_mode, full_dataset.transform = True, qsm_aug
    for pt_epoch in range(5):
        model.base_model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            optimizer_pt.zero_grad()
            
            # Use visual features for reconstruction
            _, feats = model(imgs, clin)
            recon = decoder(feats)
            
            loss = criterion_pt(recon, imgs)
            loss.backward()
            optimizer_pt.step()

    # --- B. FINE-TUNING ---
    print(">>> Fine-tuning Classifier Head (Backbone Frozen)")
    for param in model.base_model.parameters(): param.requires_grad = False
    
    optimizer = torch.optim.Adam(model.fusion.parameters(), lr=5e-5, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)

    best_f1, patience, best_metrics_this_split = 0, 0, None
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(30):
        model.train(); full_dataset.train_mode = True
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits, _ = model(imgs, clin)
            loss_fn(logits, lbls).backward(); optimizer.step()
        
        model.eval(); full_dataset.train_mode = False
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        current_f1 = 2*(m[2]*m[3])/(m[2]+m[3]) if (m[2]+m[3])>0 else 0

        if current_f1 > best_f1:
            best_f1, best_metrics_this_split, patience = current_f1, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/best_f1_split_{split}.pth")
        else: patience += 1

        print(f"Split {split} Ep {epoch} | F1: {current_f1:.4f} | AUC: {m[5]:.4f} | Acc: {m[1]:.4f}")
        if patience >= 10: break
    
    if best_metrics_this_split is not None: all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nFINAL CV SUMMARY (BEST F1 PER SPLIT)\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping Physician
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Location
Keeping  Target
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
>>> Loading preprocessed data from cache: qsm_preprocessed_cache.pt...


/tmp/ipykernel_1814833/2116010058.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cached_data = torch.load(cache_path)


NameError: name 'report_split_stats' is not defined